# Environment Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive/')
%cd /content/drive/MyDrive/spinal-bone-feature-detection
!ls

In [ ]:
!unzip -q ./datasets/ultrasound.zip -d /tmp/ultrasound
!pip install -q torchmetrics[detection]
# !pip install -q torch_geometric

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from torchvision.models import resnet18, ResNet18_Weights
from torchvision.ops import RoIAlign, complete_box_iou_loss, box_iou
from sklearn.metrics import classification_report, precision_recall_curve, confusion_matrix, roc_curve, auc
from scipy.optimize import linear_sum_assignment
from scipy.interpolate import CubicSpline

import gc
import yaml
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from pathlib import Path
from PIL import Image

from model import MultiTaskModel
from trainers import MultiTaskModelTrainer
from loader import get_loader, concat_batch

In [ ]:
%load_ext tensorboard
!rm -rf results
plt.style.use('seaborn-v0_8')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
splits_path = 'kfolds/splits1.yaml'

# Model Training

In [ ]:
train_loader = get_loader('train', splits_path=splits_path, batch_size=32)
val_loader = get_loader('val', splits_path=splits_path, batch_size=1)
for names, images, targets in train_loader:
    print(f'Batch images shape: {images.shape}')
    for names, target in zip(names, targets):
        print(f"{names}: boxes={target['boxes'].shape}, labels={target['labels'].shape}")
    break

In [ ]:
train_loader = get_loader('train', splits_path=splits_path, batch_size=4)
val_loader = get_loader('val', splits_path=splits_path, batch_size=1)
model = MultiTaskModel(criterion_box=complete_box_iou_loss, device=device)
optimizer = optim.AdamW(model.parameters(), lr=2e-4)
# scheduler = optim.lr_scheduler.OneCycleLR(optimizer, max_lr=3e-4, epochs=1000, steps_per_epoch=len(train_loader), pct_start=0.1)

In [ ]:
%%time
gc.collect()
torch.cuda.empty_cache()
trainer = MultiTaskModelTrainer(model, train_loader, val_loader, num_epochs=1000, optimizer=optimizer, scheduler=None)
trainer.train()

In [ ]:
%tensorboard --logdir results

# Evaluation

## mAP, Cls Report, PR Curve

In [ ]:
checkpoint = torch.load('best_model.pth')
trainer.model.load_state_dict(checkpoint)
mAP_preds, mAP_targets, results = trainer.evaluate()
results

In [ ]:
def plot_cf_matrix(cf_matrix, label_names, ax=None):
    labels = np.asarray([f'{name}\n{count:,}\n{percent:.2%}' for name, count, percent in zip(
        ['True Neg', 'False Pos', 'False Neg', 'True Pos'], # Group names
        cf_matrix.flatten(), # Group counts
        cf_matrix.flatten() / np.sum(cf_matrix) # Group percentages
    )]).reshape(2, 2)
    sns.heatmap(
        cf_matrix, fmt='', annot=labels,
        cmap='YlGnBu', square=True, annot_kws={'size': 12},
        xticklabels=label_names, yticklabels=label_names, ax=ax
    )

all_preds, all_targets = [], []
for pred, target in zip(mAP_preds, mAP_targets):
    all_preds.extend(pred['labels'].cpu().numpy())
    all_targets.extend(target['labels'].cpu().numpy())

In [ ]:
label_names=['Thoracic', 'Lumbar']
print(classification_report(all_targets, all_preds, target_names=label_names))
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
cm = confusion_matrix(all_targets, all_preds)
plot_cf_matrix(cm, label_names=label_names)

plt.subplot(1, 2, 2)
fpr, tpr, _ = roc_curve(all_targets, all_preds)
plt.plot(fpr, tpr, label=f'ROC curve (AUC = {auc(fpr, tpr):.3f})')
plt.plot([0, 1], [0, 1], 'm--')

plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic')
plt.legend(loc='lower right', frameon=True, shadow=True, borderpad=0.5)
plt.show()

In [ ]:
def compute_pr_curve(preds, targets, iou_threshold=0.5): # Evaluate predictions and compute PR curve
    all_scores, all_labels = [], [] # 1 for TP, 0 for FP
    for pred, target in zip(preds, targets):
        pred_boxes = pred['boxes']   # [N, 4]
        scores = pred['scores']      # [N]
        pred_labels = pred['labels'] # [N]
        gt_boxes = target['boxes']   # [M, 4]
        gt_labels = target['labels'] # [M]
        if len(pred_boxes) == 0 or len(gt_boxes) == 0: continue

        # Compute IoU between all pred and gt boxes
        iou = box_iou(pred_boxes, gt_boxes).cpu().numpy()  # [N, M]
        max_iou = np.max(iou, axis=1)  # Best IoU for each prediction
        matched_idx = np.argmax(iou, axis=1)

        # For each prediction, determine if it's a TP or FP
        for i, (score, pred_label) in enumerate(zip(scores, pred_labels)):
            if max_iou[i] >= iou_threshold:
                gt_idx = matched_idx[i]
                if pred_label == gt_labels[gt_idx]: all_labels.append(1)  # True Positive
                else: all_labels.append(0)  # False Positive (wrong label)
            else: all_labels.append(0)  # False Positive (no match)
            all_scores.append(score.cpu().numpy())

    # Sort by scores in descending order
    all_scores = np.array(all_scores)
    all_labels = np.array(all_labels)
    order = np.argsort(all_scores)[::-1]
    all_scores = all_scores[order]
    all_labels = all_labels[order]

    # Compute precision and recall
    precision, recall, _ = precision_recall_curve(all_labels, all_scores)
    return precision, recall

precision, recall = compute_pr_curve(mAP_preds, mAP_targets)
plt.plot(recall, precision, marker='.')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.grid(True)
plt.show()

## Display Validation Inference

In [ ]:
checkpoint = torch.load('best_model.pth')
model = MultiTaskModel(device=device)
model.load_state_dict(checkpoint)
model.eval()

with open('config.yaml', 'r') as f:
    config = yaml.safe_load(f)
    images_dir = Path(config['data']['images_dir'])
    processed_height, processed_width = config['preprocessing']['image_size']

def get_scale(image):
    orig_width, orig_height = image.size
    scale_x = orig_width / processed_width
    scale_y = orig_height / processed_height
    return scale_x, scale_y

In [ ]:
fig, axes = plt.subplots(nrows=2, ncols=4, figsize=(10, 15))
axes = axes.flatten()  # Flatten the grid for easy indexing
for ax in axes: ax.grid(False) # Turn off gridlines
grid_idx, pred_prid_idx = 0, 0 # Counter for the grid positions

for names, images, targets in val_loader:
    images, targets, boxes_by_images, labels_by_images, all_boxes, all_labels = concat_batch(images, targets, device)
    pred_boxes, pred_labels = model(images, boxes_by_images, all_boxes, all_labels)
    pred_labels = F.softmax(pred_labels, dim=-1)
    scores, pred_labels_idxs = pred_labels.max(dim=1)

    # Reshape to match boxes_by_images shape
    pred_boxes_by_images = torch.split(pred_boxes, [len(b) for b in boxes_by_images])
    pred_labels_by_images = torch.split(pred_labels_idxs, [len(b) for b in boxes_by_images])
    scores_by_images = torch.split(scores, [len(b) for b in boxes_by_images])

    # Plot each ground truth box
    for name, boxes, labels in zip(names, boxes_by_images, labels_by_images):
        if grid_idx >= len(axes): break # Stop if all subplots are filled
        image = Image.open(images_dir / f'{name}.jpg')
        scale_x, scale_y = get_scale(image) # Scale the boxes back to the original image dimensions
        orig_width, orig_height = image.size

        # Display image
        axes[grid_idx].imshow(image, cmap='gray')
        axes[grid_idx].axis('off') # Turn off axis labels and ticks
        axes[grid_idx].set_title(f'{orig_height} x {orig_width}\n({len(boxes)} boxes detected)')
        grid_idx += 1  # Move to the next subplot position

        for box, label in zip(boxes, labels):
            if label >= 2: continue
            x1, y1, x2, y2 = box.cpu().numpy()
            x1, y1, x2, y2 = x1 * scale_x, y1 * scale_y, x2 * scale_x, y2 * scale_y
            color = 'red' if label == 0 else 'green'
            rect = patches.Rectangle((x1, y1), x2 - x1, y2 - y1, linewidth=1, edgecolor=color, facecolor='none')
            axes[grid_idx - 1].add_patch(rect) # Add the rectangle to the plot

    # Plot each predicted box (Similar processing as for ground truth boxes)
    for name, boxes, labels, scores in zip(names, pred_boxes_by_images, pred_labels_by_images, scores_by_images):
        if pred_prid_idx >= len(axes): break # Stop if all subplots are filled
        image = Image.open(images_dir / f'{name}.jpg')
        scale_x, scale_y = get_scale(image) # Scale the boxes back to the original image dimensions
        pred_prid_idx += 1  # Move to the next subplot position

        for box, label, score in zip(boxes, labels, scores):
            if label >= 2: continue
            x1, y1, x2, y2 = box.cpu().detach().numpy() # Detach the tensor before converting to NumPy
            x1, y1, x2, y2 = x1 * scale_x, y1 * scale_y, x2 * scale_x, y2 * scale_y
            color = 'cyan' if label == 0 else 'gold'
            rect = patches.Rectangle((x1, y1), x2 - x1, y2 - y1, linewidth=1, edgecolor=color, facecolor='none')
            axes[pred_prid_idx - 1].add_patch(rect) # Add the rectangle to the plot
            axes[pred_prid_idx - 1].text(
                x1, y1, f'{score.cpu().detach().numpy():.4f}',
                fontsize=9, fontweight='bold',
                color=color, # bbox=dict(facecolor='white', alpha=0.2)
            )
    if grid_idx >= len(axes): break # Stop if all subplots are filled

for ax in axes[grid_idx:]: ax.axis('off') # Hide any unused subplots
plt.tight_layout()
plt.show()